The goal of this notebook is to explore why the python file in this repository can't handle the TEI.2 files in the perseus DL. 

To start, I want to import the function from there and inspect it.

In [2]:
from postag_perseusDL import csv_postag

from inspect import *

print(getdoc(csv_postag))

Docstring for csv_postag NEEDSDOC

:param path: An optional path of the file to write manually
:type path: str
:param skip_finished: NEEDSDOC
:type skip_finished: bool


Lets say I want to get the source code of the function:

In [3]:
print(getsource(csv_postag))

def csv_postag(path="", skip_finished=True) -> None:
    """
    Docstring for csv_postag NEEDSDOC

    :param path: An optional path of the file to write manually
    :type path: str
    :param skip_finished: NEEDSDOC
    :type skip_finished: bool
    """
    paths: list = __get_paths(path)

    """
    2026-02-15 11:46:56 INFO: Using device: cpu
    2026-02-15 11:46:56 INFO: Loading: tokenize
    2026-02-15 11:46:59 INFO: Loading: mwt
    2026-02-15 11:46:59 INFO: Loading: pos
    2026-02-15 11:47:01 INFO: Loading: lemma
    2026-02-15 11:47:02 INFO: Loading: depparse
    """
    # Add this so we can use the GPU
    # NOTE !!! USING THE GPU REQUIRED ME TO INSTALL TORCH THROUGH PIP WITH CUDA 11.8 (SEE INSTALL INSTRUCTIONS ON THEIR WEBSITE FOR MORE)
    custom_pipeline = stanza.Pipeline(
        "la", processors="tokenize,mwt,pos,lemma,depparse", use_gpu=True
    )
    # OLD CLTK: nlp = NLP(backend="stanza", custom_pipeline=custom_pipeline)

    sPathsRemoved = []

    # If we want to 

Okay, we have the source code to look at now.

Let's look closer at the following lines of code

parser: etree.XMLParser = etree.XMLParser(resolve_entities=False)
            tree: etree.ElementTree = etree.parse(
                p, parser
            )  # Use path 22 for debugging

            # Get the root of the tree. This variable will eventually hold the tei:body element
            body: etree._Element = tree.getroot()

            authority_dict = get_title_auth_body(body)


In [4]:
from lxml import etree

from postag_perseusDL import *

p = "./../canonical-latinLit/data/phi0428/phi001/phi0428.phi001.perseus-lat1.xml"

parser: etree.XMLParser = etree.XMLParser(resolve_entities=False)
tree: etree.ElementTree = etree.parse(
    p, parser
)  # Use path 22 for debugging

# Get the root of the tree. This variable will eventually hold the tei:body element
body: etree._Element = tree.getroot()

authority_dict = get_title_auth_body(body)



In [5]:
print(authority_dict)

{'body': <Element body at 0x18f06805700>, 'title': 'Bellum Alexandrinum', 'author': 'Anonymous'}


In [8]:
print(
    (
        etree.tostring(
            authority_dict['body'], 
            pretty_print=True
        )
    ).decode('utf-8')[1:1000]
)

body>
<pb id="p.10"/>
<div1 type="chapter" n="1"><p>
Bello Alexandrino conflato Caesar Rhodo atque ex Syria Cilieiaque omnem classem arcessit; Creta sagittarios, equites ab rege Nabataeorum Maleho evocat; tormenta undique conquiri et frumentum mitti, auxilia adduci iubet. Interim munitiones cotidie operibus augentur atque omnes oppidi partes, quae minus esse firmae videntur, testudinibus ac musculis aptantur;1 ex aedificiis autem per foramina in proxima aedificia arietes immittuntur, quantumque aut ruinis deicitur aut per vim recipitur loci, in tantum munitiones proferuntur. Nam incendio fere tuta est Alexandrea, quod sine contignatione ac materia sunt aedificia et structuris ac fornicibus continentur tectaque sunt rudere aut pavimentis. Caesar maxime studebat ut, quam angustissimam partem oppidi palus a meridie interiecta efficiebat, hanc operibus vineisque agendis ab reliqua parte urbis excluderet, illud spectans primum ut, cum in duas partis esset urbis divisa acies, uno consilio a


Okay, so the issue isn't with retrieving the text, which is still successful.

It must be in the get_text() function, whose source code is here:

In [9]:
print(getsource(get_text))

def get_text(element) -> str:
    """
    This function takes an element and extracts only the plaintext. This is necessary to avoid including text in notes or
    emendations. This also makes sure that text in the tail of one of those invalid tags gets kept and not completely lost.
    """
    string = ""
    sTail: str = element.tail
    sText: str = element.text
    parent: etree._Element = element.getparent()
    pText: str = parent.text
    pTail: str = parent.tail

    if pTail:
        if has_tail(parent):
            return string

    tagValid = is_valid_tag(element)
    if tagValid == 1:
        if sText:
            string += sText

    # Make sure it has a tail with alphanumeric characters in it and that it isn't inside of an invalid tag
    if (has_tail(element)) and (tagValid == 1):
        children: list = [x for x in element.iterdescendants() if (x.tail or x.text)]
        for child in children:
            if is_valid_tag(child):
                string += ("" if child.

In [11]:
#Let's go one by one

body = authority_dict['body']

max_iter = 10 #I only want to go through the first 10 as an example
iter = 0
for element in body.iter():
    iter += 1
    print(get_text(element))
    if iter == max_iter:
        break









Bello Alexandrino conflato Caesar Rhodo atque ex Syria Cilieiaque omnem classem arcessit; Creta sagittarios, equites ab rege Nabataeorum Maleho evocat; tormenta undique conquiri et frumentum mitti, auxilia adduci iubet. Interim munitiones cotidie operibus augentur atque omnes oppidi partes, quae minus esse firmae videntur, testudinibus ac musculis aptantur;1 ex aedificiis autem per foramina in proxima aedificia arietes immittuntur, quantumque aut ruinis deicitur aut per vim recipitur loci, in tantum munitiones proferuntur. Nam incendio fere tuta est Alexandrea, quod sine contignatione ac materia sunt aedificia et structuris ac fornicibus continentur tectaque sunt rudere aut pavimentis. Caesar maxime studebat ut, quam angustissimam partem oppidi palus a meridie interiecta efficiebat, hanc operibus vineisque agendis ab reliqua parte urbis excluderet, illud spectans primum ut, cum in duas partis esset urbis divisa acies, uno consilio atque imperio administraretur, deinde ut labora

Okay....so, there's no issue with getting the text out....hm. Let's create a new function based on csv_postag, with just the text extraction part. We'll print some more results to see where things go wrong.

In [30]:
from postag_perseusDL import *

def csv_postag_debug(path="", skip_finished=True) -> None:
    """
    Docstring for csv_postag NEEDSDOC

    :param path: An optional path of the file to write manually
    :type path: str
    :param skip_finished: NEEDSDOC
    :type skip_finished: bool
    """
    paths: list = __get_paths(path)
    print(f"Paths to extract: {paths}")

    """
    2026-02-15 11:46:56 INFO: Using device: cpu
    2026-02-15 11:46:56 INFO: Loading: tokenize
    2026-02-15 11:46:59 INFO: Loading: mwt
    2026-02-15 11:46:59 INFO: Loading: pos
    2026-02-15 11:47:01 INFO: Loading: lemma
    2026-02-15 11:47:02 INFO: Loading: depparse
    """
    # Add this so we can use the GPU
    # NOTE !!! USING THE GPU REQUIRED ME TO INSTALL TORCH THROUGH PIP WITH CUDA 11.8 (SEE INSTALL INSTRUCTIONS ON THEIR WEBSITE FOR MORE)
    custom_pipeline = stanza.Pipeline(
        "la", processors="tokenize,mwt,pos,lemma,depparse", use_gpu=True
    )
    # OLD CLTK: nlp = NLP(backend="stanza", custom_pipeline=custom_pipeline)

    sPathsRemoved = []

    # If we want to keep a line, we place it in the temp file.
    #with open(
    #    results_file, "r", encoding="utf-8", errors="replace", newline=""
    #) as f_read:
    #    s: set = set(f_read.read().splitlines())
    #    with open(
    #        "./temp.csv", "w", encoding="utf-8", errors="replace", newline=""
    #    ) as f_write:
    #        # If we're skipping already finished ones, lets remove paths
    #        # that we're not going to parse
    #        if skip_finished:
    #            for p in paths:
    #                if __in_file(p, s):
    #                    sPathsRemoved.append(p)
    #                    paths.remove(p)
    #        else:
    #            wr = csv.writer(f_write)
    #            f_read.seek(0)  # just to be sure
    #            read = csv.reader(f_read)
    #            for l in read:
    #                if l[2] not in paths:
    #                    wr.writerow(l)
    #    if not skip_finished:
    #        shutil.copyfile("temp.csv", results_file)
    #        os.remove("./temp.csv")

    # Get the csv.writer
    with open(results_file, "a", encoding="utf-8", errors="replace", newline="") as f:
        writer = csv.writer(f, escapechar="#")

        l_paths = len(paths)

        s_docs = []
        for p in paths:
            i_paths = paths.index(p)

            print(f"Completion of TEI text extraction: {i_paths}/{l_paths}")
            parser: etree.XMLParser = etree.XMLParser(resolve_entities=False)
            tree: etree.ElementTree = etree.parse(
                p, parser
            )  # Use path 22 for debugging

            #DEBUG
            print(etree.tostring(tree).decode('utf-8')[1:300]) 

            # Get the root of the tree. This variable will eventually hold the tei:body element
            body: etree._Element = tree.getroot()

            authority_dict = get_title_auth_body(body)

            print(authority_dict)
            

            body = authority_dict["body"]
            titleString = authority_dict["title"]
            authorString = authority_dict["author"]

            # Remember, when there are potentially Greek characters, we need encoding set to utf-8.
            # This debug file will store the results before the final version of the program

            """debug: io.TextIOWrapper = open(
                "C:/Users/T470s/Documents/GitHub/cltk-2025-atticus/perseus-debug.txt",
                "w",
                encoding="utf-8",
            )
            debug.write(str(p))"""

            #if len(body):
            #    body = body[0]

            print(f"Body: {body}")

            # Now we have the <body> element, let's get the text######################3

            # Add the text for each element, using the get_text() function
            string: str = ""
            for element in body.iter():
                string += get_text(element)

            print(f"Final string: {string[0:100]}")

            string = re.sub("\t", "", string)

            s_final_body = remove_invalid_characters(string)

            # DEBUG
            save_output(text=s_final_body)

            print(s_final_body[1:200])

csv_postag_debug(p)

Paths to extract: ['./../canonical-latinLit/data/phi0428/phi001/phi0428.phi001.perseus-lat1.xml']


2026-04-07 11:08:12 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


2026-04-07 11:08:13 INFO: Downloaded file to C:\Users\T470s\stanza_resources\resources.json
2026-04-07 11:08:14 INFO: Loading these models for language: la (Latin):
| Processor | Package       |
-----------------------------
| tokenize  | ittb          |
| mwt       | ittb          |
| pos       | ittb_nocharlm |
| lemma     | ittb_nocharlm |
| depparse  | ittb_nocharlm |

2026-04-07 11:08:14 WARNING: GPU requested, but is not available!
2026-04-07 11:08:14 INFO: Using device: cpu
2026-04-07 11:08:14 INFO: Loading: tokenize
2026-04-07 11:08:14 INFO: Loading: mwt
2026-04-07 11:08:14 INFO: Loading: pos
2026-04-07 11:08:16 INFO: Loading: lemma
2026-04-07 11:08:17 INFO: Loading: depparse
2026-04-07 11:08:18 INFO: Done loading processors!


Completion of TEI text extraction: 0/1
!DOCTYPE TEI.2 PUBLIC "-//TEI P4//DTD Main DTD Driver File//EN" "http://www.tei-c.org/Guidelines/DTD/tei2.dtd" [
<!ENTITY % TEI.XML "INCLUDE">
<!ENTITY % PersProse PUBLIC "-//Perseus P4//DTD Perseus Prose//EN" "http://www.perseus.tufts.edu/DTD/1.0/PersProse.dtd">
]>
<TEI.2>
  <teiHeader>
    <fileD
{'body': <Element body at 0x18f42daf840>, 'title': 'Bellum Alexandrinum', 'author': 'Anonymous'}
Body: <Element body at 0x18f42daf840>
Final string: 




Bello Alexandrino conflato Caesar Rhodo atque ex Syria Cilieiaque omnem classem arcessit; Creta
Bello Alexandrino conflato Caesar Rhodo atque ex Syria Cilieiaque omnem classem arcessit; Creta sagittarios, equites ab rege Nabataeorum Maleho evocat; tormenta undique conquiri et frumentum mitti, a


Okay, commenting out this code seemed to fix it:

if len(body):
    body = body[0]

I was worried about multiple "body" elements matching, but I forgot how lxml works. 
Each element is a "list" of of subelements, so of course this code would get a subelement instead of the whole body. Don't know why this didn't break earlier.

Let's try this again, with the correction (DON'T RUN THIS AGAIN, IF YOU DO, CHANGE SKIP_FINISHED TO FALSE)

In [ ]:
from postag_perseusDL import csv_postag

prefix = "./../canonical-latinLit/data/"
paths = [
    f"{prefix}phi0448/phi002/phi0448.phi002.perseus-lat3.xml", 
    f"{prefix}phi0426/phi001/phi0426.phi001.perseus-lat1.xml", 
    f"{prefix}phi0430/phi001/phi0430.phi001.perseus-lat1.xml"
]
csv_postag(paths, skip_finished = False)